<a href="https://colab.research.google.com/github/srkprattipati/Bank-marketing_ML_assignment_2/blob/project/ML_assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import os

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv("/content/bank-full.csv", sep=";")
df.columns = df.columns.str.replace('"', '')
df = df.applymap(lambda x: x.replace('"', '') if isinstance(x, str) else x)
df["y"] = df["y"].map({"yes": 1, "no": 0})

df = pd.get_dummies(df)

# -----------------------------
# 2. Split features/target
# -----------------------------
X = df.drop("y", axis=1)
y = df["y"]

# -----------------------------
# 3. Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 4. SMOTE
# -----------------------------
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

# -----------------------------
# 5. Scaling
# -----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# 6. Model folder
# -----------------------------
os.makedirs("model", exist_ok=True)

models = {}

models["Logistic Regression"] = LogisticRegression(
    max_iter=5000, solver="liblinear", class_weight="balanced"
)

models["Decision Tree"] = DecisionTreeClassifier(
    random_state=42, class_weight="balanced"
)

models["Random Forest"] = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)

models["kNN"] = KNeighborsClassifier(n_neighbors=7)

models["Naive Bayes"] = GaussianNB()

models["XGBoost"] = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

# -----------------------------
# 7. Train + Print Tables
# -----------------------------
all_metrics = []

for name, model in models.items():
    print(f"\n==============================")
    print(f"📌 Training: {name}")
    print("==============================")

    # kNN uses scaled data
    if name == "kNN":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    mcc = matthews_corrcoef(y_test, preds)

    # Print table for each model
    model_table = pd.DataFrame({
        "Metric": ["Accuracy", "AUC Score", "Precision", "Recall", "F1 Score", "MCC Score"],
        "Value": [accuracy, auc, precision, recall, f1, mcc]
    })

    print(model_table.to_string(index=False))

    # Save model
    joblib.dump(model, f"model/{name.replace(' ', '_').lower()}.joblib")

    # Append for comparison table
    all_metrics.append([name, accuracy, auc, precision, recall, f1, mcc])

# -----------------------------
# 8. Final Comparison Table
# -----------------------------
comparison_df = pd.DataFrame(
    all_metrics,
    columns=["Model", "Accuracy", "AUC Score", "Precision", "Recall", "F1 Score", "MCC Score"]
)

print("\n\n==============================")
print("📊 FINAL MODEL COMPARISON TABLE")
print("==============================\n")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("model_metrics.csv", index=False)

print("\n🎉 Training complete. All models and metrics saved.")


/tmp/ipykernel_1094/3258516559.py:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('"', '') if isinstance(x, str) else x)



📌 Training: Logistic Regression
   Metric    Value
 Accuracy 0.902134
AUC Score 0.901702
Precision 0.639291
   Recall 0.375236
 F1 Score 0.472901
MCC Score 0.441252

📌 Training: Decision Tree
   Metric    Value
 Accuracy 0.873493
AUC Score 0.704111
Precision 0.461191
   Recall 0.482987
 F1 Score 0.471837
MCC Score 0.400159

📌 Training: Random Forest
   Metric    Value
 Accuracy 0.907110
AUC Score 0.928052
Precision 0.646900
   Recall 0.453686
 F1 Score 0.533333
MCC Score 0.492908

📌 Training: kNN
   Metric    Value
 Accuracy 0.892845
AUC Score 0.812171
Precision 0.582868
   Recall 0.295841
 F1 Score 0.392476
MCC Score 0.364185

📌 Training: Naive Bayes
   Metric    Value
 Accuracy 0.841756
AUC Score 0.741404
Precision 0.357959
   Recall 0.444234
 F1 Score 0.396457
MCC Score 0.308975

📌 Training: XGBoost
   Metric    Value
 Accuracy 0.908548
AUC Score 0.935055
Precision 0.625954
   Recall 0.542533
 F1 Score 0.581266
MCC Score 0.531937


📊 FINAL MODEL COMPARISON TABLE

              Mode